# Notebook 11 — Beyond Training: Regularization Tricks & Inference Acceleration

Your model is trained. How do you squeeze out the last point of accuracy? How do you run it 1.5-5x faster in production, shrink it 4x on disk, and prove it still works? This notebook collects the high-leverage tricks every production ML engineer reaches for:

- **EMA** / **SWA** — free accuracy from weight averaging.
- **Test-Time Augmentation (TTA)** — another free point by averaging augmented predictions.
- **AMP** (mixed precision) and **`torch.compile`** — training and inference speed-ups.
- **ONNX export** — portable model format for other runtimes.
- **INT8 quantization** — 4x smaller, often faster, on CPU.
- A **benchmark table** comparing all the options.

## Learning goals
- Implement an `EMAModel` from scratch and train with it.
- Use `torch.optim.swa_utils` for SWA.
- Implement a simple TTA wrapper.
- Add AMP to a training loop and measure the speed-up.
- Export a model to ONNX and validate the output matches PyTorch.
- Quantize a model to INT8 and measure disk / latency trade-offs.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/11_regularization_and_inference.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## 1. A small model + dataset to experiment on

We'll use CIFAR-10 resized to 96x96 with a small timm EfficientNetV2 (`tf_efficientnetv2_b0`). Small enough to train on CPU in a few minutes, large enough that EMA / AMP / ONNX all make a visible difference.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import timm
import numpy as np, time, copy
from utils.env import data_dir, checkpoints_dir

SIZE = 96
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize(SIZE + 8),
    transforms.RandomCrop(SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize(SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
full_train = torchvision.datasets.CIFAR10(data_dir(), train=True,  download=True, transform=train_tf)
full_test  = torchvision.datasets.CIFAR10(data_dir(), train=False, download=True, transform=eval_tf)

# Subset for speed: 5000 train / 1000 val.
rng = np.random.default_rng(0)
train_idx = rng.choice(len(full_train), size=5000, replace=False)
val_idx   = rng.choice(len(full_test),  size=1000, replace=False)
train_ds  = Subset(full_train, train_idx.tolist())
val_ds    = Subset(full_test,  val_idx.tolist())
print(f'train={len(train_ds)}, val={len(val_ds)}')

def build_model():
    torch.manual_seed(0)
    return timm.create_model('tf_efficientnetv2_b0', pretrained=True, num_classes=10).to(device)

def make_loaders(bs=64):
    return (DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=0),
            DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0))

def evaluate(m, dl):
    m.eval(); correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (m(xb).argmax(1) == yb).sum().item(); total += yb.numel()
    return correct / total

## 2. Exponential Moving Average (EMA) of weights

During training, the model's weights bounce around the loss landscape. EMA keeps a *shadow copy* that is a smoothed version of the weights:

$$\theta^{\mathrm{EMA}} \leftarrow d \cdot \theta^{\mathrm{EMA}} + (1 - d) \cdot \theta$$

where $d$ (decay) is close to 1 — typically 0.999 for short training runs, 0.9999 for long ones. Evaluate with the EMA copy. It usually buys 0.3-1% val accuracy for free and is a staple of modern training recipes (used by EfficientNet, ConvNeXt, DINO, YOLO, ...).

In [ ]:
class EMAModel:
    """Simple EMA of model parameters + buffers. Call .update(model) every optim step."""
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        # We keep the EMA weights in a no-grad deep copy, on the same device.
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self.decay
        # parameters
        for ep, p in zip(self.module.parameters(), model.parameters()):
            ep.mul_(d).add_(p.detach(), alpha=1 - d)
        # buffers (BN running stats, etc.) -- copy outright, they are not trained.
        for eb, b in zip(self.module.buffers(), model.buffers()):
            eb.copy_(b)

    def eval_model(self) -> nn.Module:
        return self.module

In [ ]:
model = build_model()
ema = EMAModel(model, decay=0.999)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
train_dl, val_dl = make_loaders()

for epoch in range(3):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward(); opt.step()
        ema.update(model)                      # the key new line
    acc_raw = evaluate(model, val_dl)
    acc_ema = evaluate(ema.eval_model(), val_dl)
    print(f'epoch {epoch+1}: raw_val={acc_raw:.3f}   ema_val={acc_ema:.3f}   delta={acc_ema-acc_raw:+.3f}')

ACC_RAW, ACC_EMA = acc_raw, acc_ema

## 3. SWA (Stochastic Weight Averaging)

**SWA** (Izmailov et al., 2018) is EMA's blunter cousin: at the end of training, simply average weights from the last $N$ epochs. PyTorch ships `torch.optim.swa_utils.AveragedModel` (does the running average for you) and `SWALR` (a constant-or-cyclic LR schedule during the SWA phase). One extra pass with `update_bn` recomputes BatchNorm statistics.

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

model = build_model()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=3)

swa_model = AveragedModel(model)
swa_scheduler = SWALR(opt, swa_lr=5e-5)
SWA_START = 2   # start averaging after epoch 1

for epoch in range(3):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    if epoch >= SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()
    print(f'epoch {epoch+1}: raw_val={evaluate(model, val_dl):.3f}')

# One final pass to repopulate BN running stats for the averaged model.
update_bn(train_dl, swa_model, device=device)
acc_swa = evaluate(swa_model, val_dl)
print(f'SWA val accuracy: {acc_swa:.3f}')
ACC_SWA = acc_swa

## 4. Test-Time Augmentation (TTA)

At inference, feed the model **N augmented versions** of each image (original, hflip, a few crops) and average the logits (or softmax probs). Costs N× inference but usually gains 0.2-0.5% top-1 accuracy. It's cheap insurance on benchmarks and Kaggle submissions.

In [ ]:
@torch.no_grad()
def tta_predict(m, x):
    """Average logits over (original, hflip). Add more views if you have budget."""
    m.eval()
    preds = [m(x)]
    preds.append(m(torch.flip(x, dims=[-1])))      # horizontal flip
    return torch.stack(preds).mean(dim=0)

correct_plain, correct_tta, total = 0, 0, 0
for xb, yb in val_dl:
    xb, yb = xb.to(device), yb.to(device)
    correct_plain += (model(xb).argmax(1) == yb).sum().item()
    correct_tta   += (tta_predict(model, xb).argmax(1) == yb).sum().item()
    total += yb.numel()
print(f'plain val acc : {correct_plain/total:.3f}')
print(f'TTA (hflip)   : {correct_tta  /total:.3f}')
ACC_TTA = correct_tta / total

## 5. Mixed-precision training (AMP)

`torch.cuda.amp.autocast` runs most ops in **float16**/bfloat16 on Tensor-Core GPUs. `GradScaler` keeps the gradient scale high so fp16 underflow doesn't kill training. Expect 1.5-2× speed-up and ~40% less memory on Ampere/Ada GPUs. Tiny accuracy change, if any.

The pattern is three lines inside your loop:

```python
with torch.cuda.amp.autocast():
    loss = F.cross_entropy(model(xb), yb)
scaler.scale(loss).backward()
scaler.step(opt); scaler.update()
```

In [ ]:
use_amp = torch.cuda.is_available()
model_amp = build_model()
opt_amp = torch.optim.AdamW(model_amp.parameters(), lr=3e-4)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

t0 = time.time()
model_amp.train()
for xb, yb in train_dl:
    xb, yb = xb.to(device), yb.to(device)
    opt_amp.zero_grad()
    with torch.cuda.amp.autocast(enabled=use_amp):
        loss = F.cross_entropy(model_amp(xb), yb)
    scaler.scale(loss).backward()
    scaler.step(opt_amp); scaler.update()
print(f'AMP={use_amp}: one epoch in {time.time()-t0:.1f}s')

## 6. `torch.compile` (PyTorch 2.0+)

`torch.compile(model)` traces the forward pass into a fused graph (via TorchDynamo + Inductor), typically 10-30% faster on both training and inference — no code changes beyond the one line. First call is slow because it JIT-compiles; subsequent calls are fast.

Caveats: some exotic ops still fall back to eager mode; debugging gets harder; on some Windows setups the backend may not be available. Wrap it in a try/except for portability.

In [ ]:
compiled_ok = False
try:
    cm = torch.compile(build_model(), mode='reduce-overhead')
    # warm-up
    with torch.no_grad():
        _ = cm(next(iter(val_dl))[0].to(device))
    compiled_ok = True
    print('torch.compile succeeded')
except Exception as e:
    print('torch.compile not available here:', e)

## 7. ONNX export

**ONNX** (Open Neural Network Exchange) is a portable model format. Once exported, the same model can run with `onnxruntime` on CPU (AVX-optimized), on GPU, on mobile (Core ML, Android NNAPI), on edge devices (TensorRT, OpenVINO). It's the default "deploy outside PyTorch" path.

`torch.onnx.export` traces the model with a dummy input. We then load via `onnxruntime` and verify the outputs match PyTorch to 1e-4.

In [ ]:
import os
cpu_model = build_model().cpu().eval()
dummy = torch.randn(1, 3, SIZE, SIZE)
onnx_path = os.path.join(checkpoints_dir(), 'effv2b0_cifar.onnx')

torch.onnx.export(
    cpu_model,
    dummy,
    onnx_path,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print('ONNX size on disk:', os.path.getsize(onnx_path) / 1e6, 'MB')

In [ ]:
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    out_onnx = sess.run(None, {'input': dummy.numpy()})[0]
    with torch.no_grad():
        out_pt = cpu_model(dummy).numpy()
    diff = np.abs(out_onnx - out_pt).max()
    print(f'max abs difference (ONNX vs PyTorch): {diff:.2e}')
    assert diff < 1e-3, 'ONNX drift is too large'
    ONNX_OK = True
except Exception as e:
    print('onnxruntime not available:', e)
    ONNX_OK = False

## 8. INT8 quantization

**Quantization** stores weights (and sometimes activations) in 8-bit integers instead of 32-bit floats. Benefits:

- **4× smaller on disk.** An 80 MB fp32 model becomes 20 MB.
- **Faster on CPU.** INT8 MMA instructions are fast on modern x86 / ARM.
- Tiny accuracy loss if done right (usually < 1%).

We use `onnxruntime.quantization.quantize_dynamic`, which is the simplest zero-calibration path: linear layers get quantized to INT8, the rest stays fp32. For CNN-heavy workloads you'd prefer *static* quantization with a calibration dataset, but dynamic is a fine start.

In [ ]:
QUANT_OK = False
if ONNX_OK:
    try:
        from onnxruntime.quantization import quantize_dynamic, QuantType
        onnx_q_path = os.path.join(checkpoints_dir(), 'effv2b0_cifar.int8.onnx')
        quantize_dynamic(onnx_path, onnx_q_path, weight_type=QuantType.QInt8)
        print('INT8 size on disk:', os.path.getsize(onnx_q_path) / 1e6, 'MB')
        sess_q = ort.InferenceSession(onnx_q_path, providers=['CPUExecutionProvider'])
        out_q = sess_q.run(None, {'input': dummy.numpy()})[0]
        print('INT8 vs fp32 max diff:', np.abs(out_q - out_pt).max())
        QUANT_OK = True
    except Exception as e:
        print('quantization path unavailable:', e)

## 9. Benchmark table — latency, size, accuracy

Finally, a head-to-head. For 100 random val images we measure wall-clock latency per image and accuracy for:
- **fp32 CPU** — portable baseline.
- **fp32 GPU** — if CUDA is available.
- **fp16 GPU (AMP autocast for inference)** — optional, CUDA-only.
- **INT8 CPU (ONNX)** — shrunk model.

Latency in ms/image and disk size in MB are the numbers your deployment team will care about.

In [ ]:
import pandas as pd

# Materialize first 100 val images / labels as numpy arrays.
xs, ys = [], []
for xb, yb in val_dl:
    xs.append(xb); ys.append(yb)
    if sum(t.size(0) for t in xs) >= 100: break
x100 = torch.cat(xs)[:100]
y100 = torch.cat(ys)[:100]

def bench_torch(m, x, n_warm=3):
    m.eval()
    with torch.no_grad():
        for _ in range(n_warm): _ = m(x[:8].to(next(m.parameters()).device))
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.time()
        pred = m(x.to(next(m.parameters()).device)).argmax(1).cpu()
        if torch.cuda.is_available(): torch.cuda.synchronize()
        dt = (time.time() - t0) / x.size(0) * 1000
    return dt, (pred == y100).float().mean().item()

rows = []
# fp32 CPU
lat, acc = bench_torch(cpu_model, x100)
rows.append(dict(setup='fp32 CPU', latency_ms=round(lat, 2), acc=round(acc, 3),
                 disk_mb=round(os.path.getsize(onnx_path)/1e6, 1)))

if torch.cuda.is_available():
    gpu_model = copy.deepcopy(cpu_model).to(device)
    lat, acc = bench_torch(gpu_model, x100)
    rows.append(dict(setup='fp32 GPU', latency_ms=round(lat, 2), acc=round(acc, 3), disk_mb=None))
    # fp16 inference via autocast
    lat0 = time.time()
    with torch.no_grad(), torch.cuda.amp.autocast():
        _ = gpu_model(x100[:8].to(device))
        torch.cuda.synchronize()
        t0 = time.time()
        pred_fp16 = gpu_model(x100.to(device)).argmax(1).cpu()
        torch.cuda.synchronize()
        lat_fp16 = (time.time() - t0) / x100.size(0) * 1000
    acc_fp16 = (pred_fp16 == y100).float().mean().item()
    rows.append(dict(setup='fp16 GPU (AMP)', latency_ms=round(lat_fp16, 2), acc=round(acc_fp16, 3), disk_mb=None))

if ONNX_OK:
    sess_cpu = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    # warm
    _ = sess_cpu.run(None, {'input': x100[:8].numpy()})
    t0 = time.time()
    logits = sess_cpu.run(None, {'input': x100.numpy()})[0]
    lat_on = (time.time() - t0) / x100.size(0) * 1000
    acc_on = (logits.argmax(1) == y100.numpy()).mean()
    rows.append(dict(setup='fp32 ONNX CPU', latency_ms=round(lat_on, 2), acc=round(float(acc_on), 3),
                     disk_mb=round(os.path.getsize(onnx_path)/1e6, 1)))

if QUANT_OK:
    _ = sess_q.run(None, {'input': x100[:8].numpy()})
    t0 = time.time()
    logits_q = sess_q.run(None, {'input': x100.numpy()})[0]
    lat_q = (time.time() - t0) / x100.size(0) * 1000
    acc_q = (logits_q.argmax(1) == y100.numpy()).mean()
    rows.append(dict(setup='INT8 ONNX CPU', latency_ms=round(lat_q, 2), acc=round(float(acc_q), 3),
                     disk_mb=round(os.path.getsize(onnx_q_path)/1e6, 1)))

bench_df = pd.DataFrame(rows)
bench_df

## Key Takeaways

- **EMA and SWA** are cheap, model-agnostic accuracy boosts — prefer EMA if you need per-epoch eval, SWA if you only need an end-of-training model.
- **TTA** is the last ~0.5% before a deadline, at the cost of N× inference. Stick to cheap augmentations (hflip, center-crop, 5-crop).
- **AMP** makes training 1.5-2× faster on modern GPUs with almost no code change. Always use it for training runs over a few minutes.
- **`torch.compile`** is a one-line speed-up on PyTorch 2.x. Worth trying; drop back to eager if a specific op misbehaves.
- **ONNX + INT8** are the usual deployment path: portable, 4× smaller, often faster on CPU. Verify numerical equivalence with a diff tolerance test.
- Always build a small **benchmark table** before deciding what to ship — latency, disk size, and accuracy are three axes, and the right trade-off depends on the product.

## Exercises

1. Sweep EMA `decay` over `[0.99, 0.999, 0.9999]`. How does the optimal value shift with the number of training steps?
2. Replace the simple `hflip` TTA with a 5-crop TTA (four corners + center). How much accuracy does it add, and how much latency?
3. Benchmark `torch.compile(mode='reduce-overhead')` vs `'max-autotune'`. Any difference on your GPU?
4. Swap `quantize_dynamic` for **static** quantization with a calibration loop of 100 images. Does accuracy improve?
5. Export the same model to **TorchScript** (`torch.jit.trace` / `torch.jit.script`) and add a row to the benchmark table. How does it compare to ONNX on CPU?
6. Deploy the ONNX model behind a tiny FastAPI endpoint and measure end-to-end latency including JSON (de)serialization. Where does time actually go?